# 🧠 MEMS ML System - LSTM Training Notebook

This notebook trains an LSTM model for **Remaining Useful Life (RUL) prediction** using Google Colab's free GPU.

## Instructions:
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Run all cells
3. Download the trained model file
4. Upload to your GitHub repository

In [ ]:
# Install required packages
!pip install tensorflow numpy -q

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import json

# Check GPU
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Generate synthetic MEMS sensor data for training
def generate_synthetic_rul_data(n_samples=5000, sequence_length=30):
    """Generate synthetic sensor data with RUL labels"""
    np.random.seed(42)
    
    # Generate sensor readings
    time = np.linspace(0, 100, n_samples)
    
    # Simulate sensor degradation
    base_signal = np.sin(2 * np.pi * 0.1 * time) * 9.81
    noise = np.random.normal(0, 0.5, n_samples)
    degradation = np.linspace(0, 2, n_samples)  # Increasing degradation
    drift = 0.01 * time
    
    signal = base_signal + noise * (1 + degradation) + drift
    
    # Create RUL labels (100% to 0%)
    rul = np.linspace(100, 0, n_samples)
    
    # Create sequences
    X, y = [], []
    for i in range(len(signal) - sequence_length):
        X.append(signal[i:i+sequence_length])
        y.append(rul[i+sequence_length])
    
    X = np.array(X).reshape(-1, sequence_length, 1)
    y = np.array(y)
    
    return X, y

# Generate data
X, y = generate_synthetic_rul_data(5000, 30)
print(f"Training data shape: X={X.shape}, y={y.shape}")

In [ ]:
# Split data
split_idx = int(len(X) * 0.8)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

In [ ]:
# Build LSTM Model
def build_lstm_model(sequence_length=30, n_features=1):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(sequence_length, n_features)),
        BatchNormalization(),
        Dropout(0.2),
        
        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.2),
        
        Dense(16, activation='relu'),
        Dense(1, activation='linear')  # RUL prediction
    ])
    
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Create model
model = build_lstm_model()
model.summary()

In [ ]:
# Train the model
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print("🚀 Starting training with GPU...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print("\n✅ Training complete!")
print(f"Final Loss: {history.history['loss'][-1]:.4f}")
print(f"Final MAE: {history.history['mae'][-1]:.4f}")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Model Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# MAE plot
axes[1].plot(history.history['mae'], label='Training MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_title('Model MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

In [ ]:
# Save the trained model
model.save('lstm_rul_model.keras')
print("✅ Model saved as 'lstm_rul_model.keras'")

# Save training history for the frontend chart
training_history = {
    'loss': [float(x) for x in history.history['loss']],
    'val_loss': [float(x) for x in history.history['val_loss']],
    'mae': [float(x) for x in history.history['mae']],
    'val_mae': [float(x) for x in history.history['val_mae']],
    'epochs': len(history.history['loss']),
    'final_loss': float(history.history['loss'][-1]),
    'final_mae': float(history.history['mae'][-1])
}

with open('training_history.json', 'w') as f:
    json.dump(training_history, f, indent=2)

print("✅ Training history saved as 'training_history.json'")

In [ ]:
# Test the model with predictions
predictions = model.predict(X_val[:10])
print("\n📊 Sample Predictions vs Actual:")
print("-" * 40)
for i in range(10):
    print(f"Predicted: {predictions[i][0]:.2f}% | Actual: {y_val[i]:.2f}%")

In [ ]:
# Download the files
from google.colab import files

print("\n📥 Download these files and upload to your GitHub repo:")
print("1. lstm_rul_model.keras - The trained model")
print("2. training_history.json - Training history for frontend")
print("3. training_history.png - Training plot")

files.download('lstm_rul_model.keras')
files.download('training_history.json')
files.download('training_history.png')

## ✅ Next Steps

After downloading the files:

1. **Upload to GitHub:**
   - Go to `https://github.com/consixdent10/mems-ml-system`
   - Upload `lstm_rul_model.keras` to `backend/saved_models/`
   - Upload `training_history.json` to `backend/saved_models/`

2. **Render will auto-redeploy** and use the pre-trained model

3. **The website will now:**
   - Show "✅ Model Pre-Trained"
   - Make instant RUL predictions
   - Display the training history chart